# Notebook 09 — Train Demand Forecaster → .pkl

Trains a demand forecasting regressor from transaction history aggregated by crop and month.

In [ ]:
import os, json, hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib

EXPORTS_DIR = Path(os.getenv('EXPORTS_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\exports\models'))
TX_CSV = Path(os.getenv('TRANSACTION_DATASET_CSV', r'C:\Users\MJ\Desktop\Agric\jupyter\data\transactions\synthetic_transactions.csv'))
MODEL_VERSION = os.getenv('MODEL_VERSION', 'v1')
ALLOW_SYNTHETIC_DATA = os.getenv('ALLOW_SYNTHETIC_DATA', 'false').lower() == 'true'
MIN_ROWS = int(os.getenv('MIN_DEMAND_ROWS', '100'))
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

In [ ]:
if TX_CSV.exists():
    tx = pd.read_csv(TX_CSV)
    dataset_source = str(TX_CSV)
elif ALLOW_SYNTHETIC_DATA:
    rng = np.random.default_rng(42)
    n = 12000
    crops = ['maize', 'soybeans', 'wheat', 'sugar_beans', 'groundnuts']
    tx = pd.DataFrame({
        'crop_type': rng.choice(crops, n),
        'quantity': rng.lognormal(4, 1.0, n),
        'price_per_kg': rng.uniform(0.05, 3.0, n),
        'created_at': pd.date_range('2022-01-01', periods=n, freq='h'),
    })
    dataset_source = 'synthetic_smoke_test'
else:
    raise FileNotFoundError(f'Demand dataset not found: {TX_CSV}')

required = {'crop_type', 'quantity', 'price_per_kg', 'created_at'}
missing = sorted(required - set(tx.columns))
if missing:
    raise ValueError(f'Demand dataset missing columns: {missing}')
if len(tx) < MIN_ROWS:
    raise ValueError(f'Demand dataset has {len(tx)} rows, minimum required is {MIN_ROWS}')

tx['created_at'] = pd.to_datetime(tx['created_at'])
tx['month_start'] = tx['created_at'].dt.to_period('M').dt.to_timestamp()
monthly = tx.groupby(['crop_type', 'month_start']).agg(demand_kg=('quantity', 'sum'), avg_price=('price_per_kg', 'mean')).reset_index()
monthly = monthly.sort_values(['crop_type', 'month_start'])
monthly['month'] = monthly['month_start'].dt.month
monthly['year'] = monthly['month_start'].dt.year
monthly['prev_demand_kg'] = monthly.groupby('crop_type')['demand_kg'].shift(1).fillna(monthly['demand_kg'].median())
crop_index = {crop: idx for idx, crop in enumerate(sorted(monthly['crop_type'].unique()))}
monthly['crop_idx'] = monthly['crop_type'].map(crop_index).astype(int)
FEATURES = ['crop_idx', 'month', 'year', 'avg_price', 'prev_demand_kg']
TARGET = 'demand_kg'
X = monthly[FEATURES].values.astype(np.float32)
y = monthly[TARGET].values.astype(np.float32)
print('Dataset source:', dataset_source)
print('Monthly rows:', len(monthly))
monthly.head()

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
model = GradientBoostingRegressor(random_state=42)
model.fit(X_train_s, y_train)
pred = model.predict(X_val_s)
mae = mean_absolute_error(y_val, pred)
r2 = r2_score(y_val, pred)
print(f'Val MAE: {mae:.2f} kg | R2: {r2:.4f}')

In [ ]:
model_path = EXPORTS_DIR / 'demand_forecaster_v1.pkl'
scaler_path = EXPORTS_DIR / 'demand_scaler_v1.pkl'
joblib.dump(model, model_path, compress=3)
joblib.dump(scaler, scaler_path, compress=3)
meta = {'model': 'demand_forecaster_v1', 'version': MODEL_VERSION, 'format': 'pkl', 'source_notebook': '09_train_demand_forecaster.ipynb', 'dataset_source': dataset_source, 'dataset_rows': int(len(tx)), 'monthly_rows': int(len(monthly)), 'features': FEATURES, 'target': TARGET, 'crop_index': crop_index, 'val_mae': round(float(mae), 4), 'val_r2': round(float(r2), 4), 'sha256': sha256_file(model_path), 'scaler_sha256': sha256_file(scaler_path), 'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
(EXPORTS_DIR / 'demand_forecaster_metadata.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))